# Notebook 01.1 — Refinamiento de Evidencia para la Tesis

Proyecto: **Reconocimiento Acústico de Vehículos de Emergencia**.

Esta notebook no repite la auditoría ni el EDA. Regenera, desde `/kaggle/input`, las cabeceras, los hashes y una muestra acústica con semilla 42. Con eso prepara las figuras y tablas de las secciones 3.3, 3.4 y 3.5. No lee los CSV de las notebooks anteriores y no modifica sus salidas.


## 00. Objetivo

Separar tres niveles: la evidencia medida, la interpretación descriptiva y la implicación para el capítulo de preparación. No hay modelo, ni métrica de clasificación, ni característica declarada como definitiva.


## 01. Configuración

Semilla 42. FFT de 2048, salto de 512, ventana Hann, 64 bandas Mel, 13 MFCC, fmin 20 Hz, fmax igual al Nyquist de cada archivo y roll-off al 85 %. El estéreo se promedia solo en memoria.


In [ ]:
from __future__ import annotations

# Notebook 01.1. No lee CSV de las notebooks 00, 00.1 ni 01.
# Regenera el inventario de cabeceras y la muestra acústica desde /kaggle/input.

import ast
import hashlib
import json
import math
import os
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

os.environ.setdefault("NUMBA_CACHE_DIR", "/tmp/numba-cache")
Path(os.environ["NUMBA_CACHE_DIR"]).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf

import librosa
import librosa.display
from matplotlib.patches import Patch

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

try:
    import openpyxl  # noqa: F401
    HAS_OPENPYXL = True
except ImportError:
    HAS_OPENPYXL = False


SEED = 42
np.random.seed(SEED)

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/dataset_eda_refined")

N_FFT = 2048
HOP_LENGTH = 512
WIN_LENGTH = 2048
WINDOW = "hann"
N_MELS = 64
N_MFCC = 13
FMIN = 20.0
ROLLOFF_PERCENT = 0.85
HASH_CHUNK = 1024 * 1024
RARE_SR_MAX_COUNT = 20
AUDIO_EXTENSIONS = {".wav", ".flac", ".mp3", ".ogg", ".m4a"}
DATASET_ORDER = ["audioset_ev", "lssiren", "sirennet", "a3siren"]
DISPLAY = {
    "audioset_ev": "AudioSet-EV v2",
    "lssiren": "LSSiren",
    "sirennet": "sireNNet",
    "a3siren": "A3Siren-Recordings",
}
CANONICAL = {
    "audioset_ev": Path("/kaggle/input/datasets/jeancdevx/audioset-ev"),
    "lssiren": Path("/kaggle/input/datasets/jeancdevx/lssiren/lssiren"),
    "sirennet": Path("/kaggle/input/datasets/jeancdevx/sirennet/sireNNet"),
    "a3siren": Path("/kaggle/input/datasets/jeancdevx/a3s-rec"),
}
SIREN_FOLDERS = ("ambulance", "firetruck", "police", "traffic")
POLARITY = ("Positive_files", "Negative_files")
PARTITIONS = ("balanced_train", "eval", "unbalanced", "unbalanced_train")
LSSIREN_FOLDERS = ("Emergency Vehicle Sirens", "Road Noises")
SAMPLE_TARGETS = {
    "audioset_positive": 200,
    "audioset_negative": 200,
    "lssiren_ambulance": 200,
    "lssiren_road": 200,
    "sirennet_per_class": 150,
}
COLORS = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]
PRIOR_OUTPUTS = ("dataset_eda", "dataset_audit", "dataset_audit_final")
STATE: dict = {}

plt.rcParams.update({
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.grid": False,
})


def configure(input_root: Path | None = None, output_root: Path | None = None, sample_targets: dict | None = None) -> None:
    """Fija rutas de esta corrida. No abre resultados de notebooks anteriores."""
    global INPUT_ROOT, OUTPUT_ROOT, SAMPLE_TARGETS
    INPUT_ROOT = Path("/kaggle/input") if input_root is None else Path(input_root)
    OUTPUT_ROOT = Path("/kaggle/working/dataset_eda_refined") if output_root is None else Path(output_root)
    if sample_targets:
        SAMPLE_TARGETS = dict(sample_targets)
    resolved = OUTPUT_ROOT.resolve()
    if INPUT_ROOT.exists() and resolved.is_relative_to(INPUT_ROOT.resolve()):
        raise RuntimeError("La salida no puede escribirse dentro de /kaggle/input.")
    if resolved.name in PRIOR_OUTPUTS:
        raise RuntimeError("Esta notebook no modifica las salidas de las notebooks anteriores.")
    for folder in ("plots", "tables", "reports"):
        (OUTPUT_ROOT / folder).mkdir(parents=True, exist_ok=True)
    STATE.clear()
    STATE["configured_at"] = datetime.now(timezone.utc).isoformat()
    STATE["seed"] = SEED
    STATE["has_openpyxl"] = HAS_OPENPYXL
    print(f"timestamp_utc: {STATE['configured_at']}")
    print(f"seed: {SEED}")
    print(f"input: {INPUT_ROOT}")
    print(f"output: {OUTPUT_ROOT}")
    print(f"openpyxl: {'disponible' if HAS_OPENPYXL else 'no disponible; la tabla 3.5 se exporta solo en CSV'}")
    print("Esta corrida no lee CSV de auditorías ni del EDA anterior, no remuestrea y no entrena modelos.")


def show_df(frame: pd.DataFrame, n: int = 12) -> None:
    print(f"filas: {len(frame)} | columnas: {len(frame.columns)}")
    if frame.empty:
        print("(tabla vacía)")
        return
    display(frame.head(n))


def save_table(frame: pd.DataFrame, filename: str) -> Path:
    path = OUTPUT_ROOT / "tables" / filename
    frame.to_csv(path, index=False)
    print(f"escrito: {path}")
    return path


def save_figure(fig: plt.Figure, filename: str) -> Path:
    path = OUTPUT_ROOT / "plots" / filename
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    if os.environ.get("AUDIT_SHOW_PLOTS", "1") != "0":
        plt.show()
    plt.close(fig)
    print(f"gráfico: {path}")
    return path


def style_ax(ax, title: str, xlabel: str, ylabel: str) -> None:
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(True, axis="y", alpha=0.25)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            block = handle.read(HASH_CHUNK)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def part_of(relative_path: str, names: tuple[str, ...]) -> str:
    folded = {name.casefold(): name for name in names}
    for part in Path(relative_path).parts:
        if part in names:
            return part
        if part.casefold() in folded:
            return folded[part.casefold()]
    return ""


def a3_base_name(filename: str) -> str:
    stem = Path(filename).stem
    lowered = stem.casefold()
    if "-ch" in lowered:
        return stem[: lowered.rfind("-ch")]
    return stem


def a3_channel(relative_path: str, filename: str) -> str:
    for part in Path(relative_path).parts:
        token = part.casefold()
        if token.startswith("ch") and token[2:].isdigit():
            return token
    lowered = Path(filename).stem.casefold()
    if "-ch" in lowered:
        suffix = lowered.rsplit("-ch", 1)[1]
        if suffix.isdigit():
            return f"ch{int(suffix)}"
    return ""


def a3_class(relative_path: str) -> str:
    for part in Path(relative_path).parts:
        if part.casefold() in {"siren", "noise"}:
            return "Siren" if part.casefold() == "siren" else "Noise"
    return ""


In [ ]:
def discover_roots(input_root: Path) -> dict[str, Path]:
    found = {key: [] for key in DATASET_ORDER}
    for key, path in CANONICAL.items():
        if path.exists():
            found[key].append(path)
    if input_root.exists():
        for dirpath, dirnames, filenames in os.walk(input_root, followlinks=False):
            current = Path(dirpath)
            folded = {name.casefold() for name in filenames}
            children = {name.casefold() for name in dirnames}
            if {"ev_positives.csv", "ev_negatives.csv"} <= folded:
                found["audioset_ev"].append(current)
            if {"ambulance_final.csv", "road_final.csv"} <= folded:
                found["lssiren"].append(current)
            if set(SIREN_FOLDERS) <= children:
                found["sirennet"].append(current)
            name = current.name.casefold()
            if "a3s-rec" in name or "a3siren" in name:
                found["a3siren"].append(current)
    selected = {}
    for key in DATASET_ORDER:
        candidates = []
        seen = set()
        for path in found[key]:
            identity = str(path.resolve()) if path.exists() else str(path)
            if identity in seen:
                continue
            seen.add(identity)
            candidates.append(path)
        if not candidates:
            raise FileNotFoundError(f"No se encontró {DISPLAY[key]} bajo {input_root}")
        candidates.sort(key=lambda path: len(path.parts))
        selected[key] = candidates[0]
        print(f"{DISPLAY[key]}: {candidates[0]}")
    return selected


In [ ]:
def build_inventory(roots: dict[str, Path]) -> pd.DataFrame:
    rows = []
    for dataset_id, root in roots.items():
        for dirpath, dirnames, filenames in os.walk(root, followlinks=False):
            dirnames.sort()
            filenames.sort()
            for name in filenames:
                path = Path(dirpath) / name
                if path.suffix.lower() not in AUDIO_EXTENSIONS:
                    continue
                relative = path.relative_to(root).as_posix()
                info = {"sampling_rate_hz": None, "channels": None, "duration_seconds": None, "subtype": "", "read_status": "ERROR"}
                try:
                    header = sf.info(str(path))
                    info.update({
                        "sampling_rate_hz": int(header.samplerate) if header.samplerate else None,
                        "channels": int(header.channels) if header.channels else None,
                        "duration_seconds": float(header.duration) if header.duration is not None else None,
                        "subtype": str(header.subtype or ""),
                        "read_status": "OK",
                    })
                except Exception:
                    info["read_status"] = "ERROR"
                quality = "OK"
                if info["read_status"] != "OK" or not info["duration_seconds"] or info["duration_seconds"] <= 0:
                    quality = "ERROR"
                elif not info["sampling_rate_hz"] or not info["channels"]:
                    quality = "ERROR"
                rows.append({
                    "dataset_id": dataset_id,
                    "dataset": DISPLAY[dataset_id],
                    "relative_path": relative,
                    "filename": path.name,
                    "absolute_path": str(path),
                    "sampling_rate_hz": info["sampling_rate_hz"],
                    "channels": info["channels"],
                    "duration_seconds": info["duration_seconds"],
                    "subtype": info["subtype"],
                    "quality_status": quality,
                })
    frame = pd.DataFrame(rows)
    if frame.empty:
        raise RuntimeError("No se encontraron archivos de audio.")
    return frame.sort_values(["dataset", "relative_path"]).reset_index(drop=True)


def assign_hashes(frame: pd.DataFrame) -> pd.DataFrame:
    hashes = []
    for record in tqdm(frame.itertuples(index=False), total=len(frame), desc="sha256"):
        try:
            hashes.append(sha256_file(Path(record.absolute_path)))
        except Exception:
            hashes.append("")
    out = frame.copy()
    out["sha256"] = hashes
    out = out.sort_values(["dataset", "sha256", "relative_path"]).reset_index(drop=True)
    primary = []
    for _dataset, group in out.groupby("dataset", sort=False):
        seen = set()
        for digest in group["sha256"].tolist():
            if not digest or digest not in seen:
                primary.append(True)
                if digest:
                    seen.add(digest)
            else:
                primary.append(False)
    out["is_hash_primary"] = primary
    return out


def _match_column(columns: list[str], names: list[str]) -> str | None:
    folded = {str(column).casefold(): column for column in columns}
    for name in names:
        if name.casefold() in folded:
            return folded[name.casefold()]
    return None


In [ ]:
def attach_lssiren_labels(frame: pd.DataFrame, root: Path) -> pd.DataFrame:
    ambulance_path = root / "Ambulance_final.csv"
    road_path = root / "Road_final.csv"
    ambulance_map = {}
    road_map = {}
    if ambulance_path.exists():
        ambulance = pd.read_csv(ambulance_path, dtype=str)
        file_column = _match_column(list(ambulance.columns), ["filename", "file"])
        label_column = _match_column(list(ambulance.columns), ["label", "class"])
        if file_column and label_column:
            for filename, label in zip(ambulance[file_column].fillna(""), ambulance[label_column].fillna("")):
                ambulance_map[Path(str(filename)).name] = str(label).strip()
    if road_path.exists():
        road = pd.read_csv(road_path, header=None, dtype=str, keep_default_na=False)
        wav_counts = [int(road[column].astype(str).str.lower().str.endswith(".wav").sum()) for column in road.columns]
        filename_column = int(np.argmax(wav_counts)) if len(wav_counts) else 0
        label_column_road = None
        for column in road.columns:
            if int(column) == filename_column:
                continue
            values = road[column].astype(str).str.strip()
            numeric = float(values.str.match(r"^-?\d+(\.\d+)?$").mean()) if len(values) else 1.0
            if values.nunique() <= 8 and numeric < 0.5:
                label_column_road = column
                break
        if label_column_road is not None:
            for filename, label in zip(road[filename_column].astype(str), road[label_column_road].astype(str)):
                road_map[Path(filename).name] = label.strip()
    labels = []
    for record in frame.itertuples(index=False):
        if record.dataset_id != "lssiren":
            labels.append("")
            continue
        folder = part_of(record.relative_path, LSSIREN_FOLDERS)
        if folder == "Emergency Vehicle Sirens" and ambulance_map.get(record.filename):
            labels.append(ambulance_map[record.filename])
        elif folder == "Road Noises" and road_map.get(record.filename):
            labels.append(road_map[record.filename])
        elif folder == "Emergency Vehicle Sirens":
            labels.append("Ambulance")
        elif folder == "Road Noises":
            labels.append("Road")
        else:
            labels.append(folder)
    out = frame.copy()
    out["lssiren_label_raw"] = labels
    return out


def analysis_group(row: pd.Series) -> str:
    if row.dataset_id == "audioset_ev":
        return part_of(row.relative_path, POLARITY)
    if row.dataset_id == "lssiren":
        return row.lssiren_label_raw or part_of(row.relative_path, LSSIREN_FOLDERS)
    if row.dataset_id == "sirennet":
        return part_of(row.relative_path, SIREN_FOLDERS)
    if row.dataset_id == "a3siren":
        return a3_class(row.relative_path)
    return ""


def largest_remainder(sizes: list[int], sample_size: int) -> list[int]:
    total = sum(sizes)
    if total == 0 or sample_size <= 0:
        return [0 for _ in sizes]
    if sample_size >= total:
        return list(sizes)
    raw = [size / total * sample_size for size in sizes]
    quotas = [int(math.floor(value)) for value in raw]
    nonempty = [index for index, size in enumerate(sizes) if size > 0]
    if sample_size >= len(nonempty):
        for index in nonempty:
            if quotas[index] == 0:
                quotas[index] = 1
    while sum(quotas) > sample_size:
        candidates = [index for index in nonempty if quotas[index] > 1] or nonempty
        index = max(candidates, key=lambda item: quotas[item])
        quotas[index] -= 1
    remainder = sample_size - sum(quotas)
    order = sorted(range(len(sizes)), key=lambda index: (raw[index] - math.floor(raw[index]), sizes[index]), reverse=True)
    for index in order:
        if remainder <= 0:
            break
        if quotas[index] < sizes[index]:
            quotas[index] += 1
            remainder -= 1
    return quotas


def stratified_take(frame: pd.DataFrame, sample_size: int, column: str | None, force_small: str | None = None) -> pd.DataFrame:
    if frame.empty or sample_size <= 0:
        return frame.iloc[0:0].copy()
    working = frame.sort_values("relative_path").copy()
    forced = working.iloc[0:0]
    if force_small and column and column in working.columns:
        counts = working[force_small].value_counts()
        rare_values = set(counts[counts <= RARE_SR_MAX_COUNT].index)
        forced = working[working[force_small].isin(rare_values)]
    if len(forced) >= sample_size:
        return forced
    rest = working.drop(index=forced.index)
    need = sample_size - len(forced)
    if rest.empty or need <= 0:
        return forced
    generator = np.random.default_rng(SEED)
    if column is None or column not in rest.columns or rest[column].nunique(dropna=True) <= 1:
        chosen = generator.choice(len(rest), size=min(need, len(rest)), replace=False)
        return pd.concat([forced, rest.iloc[np.sort(chosen)]], ignore_index=False)
    groups = [group for _, group in rest.groupby(rest[column].astype(str), sort=True)]
    quotas = largest_remainder([len(group) for group in groups], need)
    pieces = [forced]
    for group, quota in zip(groups, quotas):
        if quota <= 0:
            continue
        chosen = generator.choice(len(group), size=min(int(quota), len(group)), replace=False)
        pieces.append(group.iloc[np.sort(chosen)])
    return pd.concat(pieces, ignore_index=False)


In [ ]:
def build_sample(frame: pd.DataFrame) -> pd.DataFrame:
    usable = frame[frame["quality_status"] == "OK"].copy()
    pieces = []
    for group_name, target in (("Positive_files", SAMPLE_TARGETS["audioset_positive"]), ("Negative_files", SAMPLE_TARGETS["audioset_negative"])):
        subset = usable[(usable["dataset_id"] == "audioset_ev") & usable["relative_path"].map(lambda path: part_of(path, POLARITY) == group_name) & usable["is_hash_primary"]].copy()
        subset["partition_folder"] = subset["relative_path"].map(lambda path: part_of(path, PARTITIONS))
        taken = stratified_take(subset, target, "partition_folder")
        taken = taken.copy()
        taken["selection_reason"] = "muestra regenerada por polaridad y carpeta de partición; un archivo por hash; seed 42"
        pieces.append(taken)
    for label, target in (("Ambulance", SAMPLE_TARGETS["lssiren_ambulance"]), ("Road", SAMPLE_TARGETS["lssiren_road"])):
        subset = usable[(usable["dataset_id"] == "lssiren") & (usable["lssiren_label_raw"] == label) & usable["is_hash_primary"]]
        taken = stratified_take(subset, target, "sampling_rate_hz", force_small="sampling_rate_hz")
        taken = taken.copy()
        taken["selection_reason"] = "muestra regenerada por clase y sampling rate; tasas con 20 archivos o menos incluidas; un archivo por hash; seed 42"
        pieces.append(taken)
    for folder in SIREN_FOLDERS:
        subset = usable[(usable["dataset_id"] == "sirennet") & usable["relative_path"].map(lambda path: part_of(path, SIREN_FOLDERS) == folder) & usable["is_hash_primary"]]
        taken = stratified_take(subset, SAMPLE_TARGETS["sirennet_per_class"], None)
        taken = taken.copy()
        taken["selection_reason"] = "muestra regenerada por carpeta; un archivo por hash; seed 42"
        pieces.append(taken)
    a3 = usable[usable["dataset_id"] == "a3siren"].copy()
    a3["selection_reason"] = "todos los canales de todos los tracks; la unidad independiente sigue siendo el track"
    pieces.append(a3)
    sample = pd.concat(pieces, ignore_index=True)
    sample["analysis_group"] = sample.apply(analysis_group, axis=1)
    sample["track_id"] = sample.apply(lambda row: a3_base_name(row.filename) if row.dataset_id == "a3siren" else "", axis=1)
    sample["logical_unit"] = sample["dataset_id"].map(lambda value: "TRACK" if value == "a3siren" else "FILE")
    return sample.sort_values(["dataset", "relative_path"]).reset_index(drop=True)


def duration_units(frame: pd.DataFrame) -> pd.DataFrame:
    pieces = []
    for dataset_id in DATASET_ORDER:
        valid = frame[(frame["dataset_id"] == dataset_id) & (frame["quality_status"] == "OK")]
        if dataset_id == "a3siren":
            grouped = valid.groupby(["track_id", "analysis_group"], as_index=False).agg(
                duration_seconds=("duration_seconds", "median"),
                sampling_rate_hz=("sampling_rate_hz", "median"),
                channels=("channels", "max"),
                relative_path=("relative_path", "first"),
                sha256=("sha256", "first"),
            )
            grouped["dataset_id"] = dataset_id
            grouped["dataset"] = DISPLAY[dataset_id]
            grouped["logical_unit"] = "TRACK"
            pieces.append(grouped)
        else:
            primary = valid[valid["is_hash_primary"]][["dataset_id", "dataset", "analysis_group", "duration_seconds", "sampling_rate_hz", "channels", "relative_path", "sha256"]].copy()
            primary["track_id"] = ""
            primary["logical_unit"] = "FILE"
            pieces.append(primary)
    return pd.concat(pieces, ignore_index=True)


In [ ]:
def load_audio(path: Path) -> tuple[np.ndarray, int, str]:
    data, samplerate = sf.read(str(path), always_2d=True, dtype="float64")
    if data.shape[1] == 1:
        return data[:, 0], int(samplerate), "mono"
    return data.mean(axis=1), int(samplerate), "mezcla_en_memoria"


def compute_features(y: np.ndarray, sr: int) -> dict:
    result = {
        "rms_mean": np.nan, "zcr_mean": np.nan, "dominant_frequency": np.nan,
        "spectral_centroid_mean": np.nan, "spectral_bandwidth_mean": np.nan,
        "spectral_rolloff_mean": np.nan, "spectral_flux_mean": np.nan, "spectral_flatness_mean": np.nan,
    }
    for index in range(1, N_MFCC + 1):
        result[f"mfcc_{index}_mean"] = np.nan
    if y.size == 0 or not sr:
        return result
    result["rms_mean"] = float(np.sqrt(np.mean(np.square(y))))
    zcr = librosa.feature.zero_crossing_rate(y, frame_length=WIN_LENGTH, hop_length=HOP_LENGTH)
    result["zcr_mean"] = float(np.mean(zcr))
    spectrum = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH, window=WINDOW))
    result["spectral_centroid_mean"] = float(np.mean(librosa.feature.spectral_centroid(S=spectrum, sr=sr)))
    result["spectral_bandwidth_mean"] = float(np.mean(librosa.feature.spectral_bandwidth(S=spectrum, sr=sr)))
    result["spectral_rolloff_mean"] = float(np.mean(librosa.feature.spectral_rolloff(S=spectrum, sr=sr, roll_percent=ROLLOFF_PERCENT)))
    result["spectral_flatness_mean"] = float(np.mean(librosa.feature.spectral_flatness(S=spectrum)))
    if spectrum.shape[1] >= 2:
        norms = spectrum / (np.sum(spectrum, axis=0, keepdims=True) + 1e-10)
        result["spectral_flux_mean"] = float(np.mean(np.linalg.norm(np.diff(norms, axis=1), axis=0)))
    frequencies = librosa.fft_frequencies(sr=sr, n_fft=N_FFT)
    mean_magnitude = spectrum.mean(axis=1)
    if mean_magnitude.size > 1:
        mean_magnitude = mean_magnitude.copy()
        mean_magnitude[0] = 0.0
        result["dominant_frequency"] = float(frequencies[int(np.argmax(mean_magnitude))])
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH, n_mels=N_MELS, fmin=FMIN, fmax=float(sr) / 2)
    for index in range(N_MFCC):
        result[f"mfcc_{index + 1}_mean"] = float(np.mean(mfcc[index]))
    return result


def extract_features(sample: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for record in tqdm(sample.itertuples(index=False), total=len(sample), desc="características"):
        row = {
            "dataset_id": record.dataset_id,
            "dataset": record.dataset,
            "analysis_group": record.analysis_group,
            "relative_path": record.relative_path,
            "absolute_path": record.absolute_path,
            "track_id": record.track_id,
            "logical_unit": record.logical_unit,
            "sampling_rate_hz": record.sampling_rate_hz,
            "channels": record.channels,
            "duration_seconds": record.duration_seconds,
            "sha256": record.sha256,
            "quality_status": record.quality_status,
            "is_hash_primary": record.is_hash_primary,
            "selection_reason": record.selection_reason,
        }
        try:
            waveform, sr, mix = load_audio(Path(record.absolute_path))
            row.update(compute_features(waveform, sr))
            row["analysis_mix"] = mix
            row["feature_error"] = ""
        except Exception as exc:
            row["analysis_mix"] = ""
            row["feature_error"] = f"{type(exc).__name__}: {exc}"
        rows.append(row)
    return pd.DataFrame(rows)


In [ ]:
def iqr_overlap(left: pd.Series, right: pd.Series) -> tuple[float, str]:
    a = pd.to_numeric(left, errors="coerce").dropna()
    b = pd.to_numeric(right, errors="coerce").dropna()
    if a.empty or b.empty:
        return float("nan"), "no hay suficientes valores para comparar"
    a1, a3 = float(a.quantile(0.25)), float(a.quantile(0.75))
    b1, b3 = float(b.quantile(0.25)), float(b.quantile(0.75))
    overlap = max(0.0, min(a3, b3) - max(a1, b1))
    span = max(a3, b3) - min(a1, b1)
    ratio = 1.0 if span <= 0 else overlap / span
    if ratio < 0.25:
        phrase = "las distribuciones se observan parcialmente diferenciadas"
    elif ratio < 0.7:
        phrase = "se observa una diferencia con solapamiento"
    else:
        phrase = "se observa solapamiento considerable"
    return ratio, phrase


def _sentence(phrases: list[str]) -> str:
    cleaned = []
    for phrase in phrases:
        text = phrase.strip().rstrip(".")
        if not text:
            continue
        cleaned.append(text[0].upper() + text[1:])
    if not cleaned:
        return "No hay otra carpeta suficiente para comparar la mediana de RMS."
    return ". ".join(cleaned) + "."


def ordered_groups(frame: pd.DataFrame, dataset_id: str) -> list[str]:
    present = [str(value) for value in frame["analysis_group"].dropna().unique()]
    if dataset_id == "sirennet":
        return [name for name in SIREN_FOLDERS if name in present]
    return sorted(present)


def draw_box(ax, frame: pd.DataFrame, column: str, groups: list[str]) -> None:
    data = []
    labels = []
    for group in groups:
        values = pd.to_numeric(frame.loc[frame["analysis_group"] == group, column], errors="coerce").dropna()
        if values.empty:
            continue
        data.append(values)
        labels.append(group)
    if not data:
        ax.set_axis_off()
        return
    ax.boxplot(data, tick_labels=labels, showfliers=False, patch_artist=True,
               boxprops={"facecolor": "#D6E3F0", "edgecolor": "#23415C"},
               medianprops={"color": "#8C2F39", "linewidth": 1.6},
               whiskerprops={"color": "#23415C"}, capprops={"color": "#23415C"})


def annotate_iqr(ax) -> None:
    ax.text(0.0, -0.22, "La caja es el rango intercuartílico. La línea central es la mediana.", transform=ax.transAxes, fontsize=8, color="#333333")


In [ ]:
configure()


## 02. Inventario regenerado

Se leen las cabeceras de los archivos y se calcula el SHA-256 para conservar un solo archivo por hash. No se recorre el audio completo de todo el corpus en esta etapa. A3Siren-Recordings queda identificado por track.


In [ ]:
def section_inventory() -> None:
    if not INPUT_ROOT.exists():
        raise RuntimeError("No existe /kaggle/input. Ejecute esta notebook en Kaggle con los cuatro datasets montados.")
    roots = discover_roots(INPUT_ROOT)
    inventory = build_inventory(roots)
    inventory = assign_hashes(inventory)
    inventory = attach_lssiren_labels(inventory, roots["lssiren"])
    inventory["analysis_group"] = inventory.apply(analysis_group, axis=1)
    inventory["track_id"] = [a3_base_name(name) if dataset_id == "a3siren" else "" for dataset_id, name in zip(inventory["dataset_id"], inventory["filename"])]
    STATE["inventory"] = inventory
    units = duration_units(inventory)
    STATE["duration_units"] = units
    save_table(units.drop(columns=[column for column in ("sha256",) if column in units.columns], errors="ignore"), "refined_duration_units.csv")
    print(f"Archivos leídos por cabecera: {len(inventory)}. Unidades de duración: {len(units)}.")
    print("El hash se calcula para no contar una copia exacta como una segunda unidad. No se vuelve a auditar el corpus.")

section_inventory()


## 03. Muestra acústica

Los descriptores se calculan solo sobre la muestra: hasta 200 archivos por polaridad en AudioSet-EV v2, hasta 200 por clase en LSSiren, hasta 150 por carpeta en sireNNet y todos los canales de A3Siren-Recordings. Las comparaciones de A3Siren usan después la mediana por track.


In [ ]:
def section_sample_features() -> None:
    sample = build_sample(STATE["inventory"])
    features = extract_features(sample)
    STATE["sample"] = sample
    STATE["features"] = features
    keep = [column for column in features.columns if column != "absolute_path"]
    save_table(features[keep], "refined_acoustic_sample.csv")
    print("Muestra acústica regenerada:")
    show_df(sample.groupby(["dataset", "analysis_group"]).size().rename("archivos").reset_index(), n=20)
    print("En A3Siren cada track aporta sus ocho canales. Las comparaciones Siren/Noise posteriores usan la mediana por track.")

section_sample_features()


## 04. Duración

La figura principal separa los archivos primarios de los tracks de A3Siren-Recordings. La caja es el rango intercuartílico y la línea central es la mediana.


In [ ]:
def section_duration_figure() -> None:
    units = STATE["duration_units"]
    file_units = units[units["logical_unit"] == "FILE"]
    track_units = units[units["logical_unit"] == "TRACK"]
    n_tracks = int(track_units["track_id"].nunique()) if "track_id" in track_units.columns else int(len(track_units))
    decision = (
        "Se seleccionó una figura de dos paneles. AudioSet-EV v2, LSSiren y sireNNet se resumen por archivo primario "
        f"({int(len(file_units))} unidades). A3Siren-Recordings se resume por track ({n_tracks} unidades). "
        "Un solo eje mezclaría archivos con tomas y sugeriría la misma unidad de análisis."
    )
    STATE["duration_decision"] = decision
    print(decision)
    fig, axes = plt.subplots(1, 2, figsize=(11.2, 5.4))
    file_order = [DISPLAY[key] for key in ("audioset_ev", "lssiren", "sirennet")]
    file_data = [pd.to_numeric(file_units.loc[file_units["dataset"] == name, "duration_seconds"], errors="coerce").dropna() for name in file_order]
    axes[0].boxplot(file_data, tick_labels=["AudioSet-EV v2", "LSSiren", "sireNNet"], showfliers=False)
    style_ax(axes[0], "Archivos primarios por hash", "Dataset", "Duración (s)")
    track_data = [pd.to_numeric(track_units["duration_seconds"], errors="coerce").dropna()]
    axes[1].boxplot(track_data, tick_labels=["A3Siren-Recordings\n(unidad: track)"], showfliers=False)
    style_ax(axes[1], "Tracks, no archivos de canal", "Dataset", "Duración (s)")
    for ax, series_list in ((axes[0], file_data), (axes[1], track_data)):
        for index, series in enumerate(series_list, start=1):
            if series.empty:
                continue
            median = float(series.median())
            ax.annotate(f"{median:.2f} s", (index, median), textcoords="offset points", xytext=(8, 4), fontsize=8, color="#333333")
    fig.suptitle("Distribución de la duración de las unidades de análisis por dataset", fontsize=13)
    fig.text(0.01, -0.02, "La caja es el rango intercuartílico. La línea central es la mediana. A3Siren-Recordings no representa 104 observaciones independientes.", fontsize=8)
    fig.tight_layout()
    save_figure(fig, "Figura_3_2_Duracion.png")
    fig, ax = plt.subplots(figsize=(7.2, 4.8))
    ax.boxplot(file_data, tick_labels=file_order, showfliers=False)
    style_ax(ax, "Duración de archivos primarios", "Dataset", "Duración (s)")
    fig.tight_layout()
    save_figure(fig, "Figura_3_2_Duracion_archivos.png")
    fig, ax = plt.subplots(figsize=(7.2, 4.8))
    siren = pd.to_numeric(track_units.loc[track_units["analysis_group"] == "Siren", "duration_seconds"], errors="coerce").dropna()
    noise = pd.to_numeric(track_units.loc[track_units["analysis_group"] == "Noise", "duration_seconds"], errors="coerce").dropna()
    ax.boxplot([series for series in (siren, noise) if not series.empty], tick_labels=[label for label, series in (("Siren", siren), ("Noise", noise)) if not series.empty], showfliers=False)
    style_ax(ax, "A3Siren-Recordings: duración por track", "Grupo observado", "Duración (s)")
    fig.tight_layout()
    save_figure(fig, "Figura_3_2_Duracion_A3_tracks.png")
    print("La figura seleccionada para la tesis es Figura_3_2_Duracion.png. Las otras dos quedan como alternativa documentada.")

section_duration_figure()


## 05. Unidades de análisis y sampling rate

Estas dos figuras describen el tamaño de cada grupo observado y las tasas presentes en los archivos utilizables.


In [ ]:
def section_structural_figures() -> None:
    units = STATE["duration_units"]
    inventory = STATE["inventory"]
    fig, axes = plt.subplots(2, 2, figsize=(11, 7.2))
    for ax, dataset_id in zip(axes.ravel(), DATASET_ORDER):
        subset = units[units["dataset_id"] == dataset_id]
        counts = subset["analysis_group"].value_counts()
        groups = ordered_groups(subset, dataset_id)
        values = [int(counts.get(group, 0)) for group in groups]
        ax.bar(groups, values, color="#4C72B0")
        unit = "tracks" if dataset_id == "a3siren" else "archivos primarios"
        style_ax(ax, f"{DISPLAY[dataset_id]} ({unit})", "Grupo observado", "Unidades de análisis")
        ax.tick_params(axis="x", rotation=20)
    fig.suptitle("Distribución de las unidades de análisis por dataset y grupo observado", fontsize=13)
    fig.tight_layout()
    save_figure(fig, "Figura_3_1_Unidades_Analisis.png")
    fig, axes = plt.subplots(2, 2, figsize=(11, 7.2))
    valid = inventory[inventory["quality_status"] == "OK"]
    for ax, dataset_id in zip(axes.ravel(), DATASET_ORDER):
        subset = valid[valid["dataset_id"] == dataset_id]
        if dataset_id != "a3siren":
            subset = subset[subset["is_hash_primary"]]
        rates = pd.to_numeric(subset["sampling_rate_hz"], errors="coerce").dropna().astype(int)
        rate_counts = rates.value_counts().sort_index()
        ax.bar([str(value) for value in rate_counts.index], rate_counts.values, color="#4C72B0")
        style_ax(ax, DISPLAY[dataset_id], "Frecuencia de muestreo (Hz)", "Archivos")
    fig.suptitle("Frecuencias de muestreo observadas en los archivos de cada corpus", fontsize=13)
    fig.tight_layout()
    save_figure(fig, "Figura_3_3_Sampling_Rate.png")

section_structural_figures()


## 06. Características espectrales de sireNNet

Cada característica conserva su escala. La lectura buscada es desplazamiento, dispersión, solapamiento y variabilidad dentro de cada carpeta.


In [ ]:
def section_sirennet_spectral() -> None:
    features = STATE["features"]
    siren = features[(features["dataset_id"] == "sirennet") & (features["feature_error"].fillna("") == "")].copy()
    groups = ordered_groups(siren, "sirennet")
    specs = (
        ("spectral_centroid_mean", "Centroide espectral (Hz)"),
        ("spectral_bandwidth_mean", "Ancho de banda (Hz)"),
        ("spectral_rolloff_mean", "Roll-off al 85 % (Hz)"),
        ("dominant_frequency", "Frecuencia dominante (Hz)"),
    )
    fig, axes = plt.subplots(2, 2, figsize=(11.2, 8.2))
    for ax, (column, ylabel) in zip(axes.ravel(), specs):
        draw_box(ax, siren, column, groups)
        style_ax(ax, ylabel.split("(")[0].strip(), "Carpeta observada", ylabel)
    fig.suptitle("Distribución de características espectrales por clase en sireNNet", fontsize=13)
    fig.text(0.01, -0.01, "Cada panel conserva su escala. La caja es el rango intercuartílico y la línea central es la mediana de la muestra acústica.", fontsize=8)
    fig.tight_layout()
    save_figure(fig, "Figura_3_7_Caracteristicas_Espectrales_sireNNet.png")
    fig, ax = plt.subplots(figsize=(7.4, 4.8))
    draw_box(ax, siren, "rms_mean", groups)
    style_ax(ax, "RMS por carpeta en sireNNet", "Carpeta observada", "RMS (amplitud)")
    fig.tight_layout()
    save_figure(fig, "Figura_3_4_RMS_sireNNet.png")

section_sirennet_spectral()


## 07. MFCC de sireNNet

La línea es la media y la banda es ±1 desviación estándar. El gráfico no asigna capacidad predictiva a ningún coeficiente.


In [ ]:
def section_mfcc() -> None:
    features = STATE["features"]
    siren = features[(features["dataset_id"] == "sirennet") & (features["feature_error"].fillna("") == "")]
    groups = ordered_groups(siren, "sirennet")
    fig, ax = plt.subplots(figsize=(9.6, 5.2))
    x = np.arange(1, N_MFCC + 1)
    for color, group_name in zip(COLORS, groups):
        subset = siren[siren["analysis_group"] == group_name]
        means = []
        stds = []
        for index in x:
            values = pd.to_numeric(subset[f"mfcc_{index}_mean"], errors="coerce").dropna()
            means.append(float(values.mean()) if not values.empty else np.nan)
            stds.append(float(values.std(ddof=1)) if len(values) >= 2 else 0.0)
        means = np.asarray(means)
        stds = np.asarray(stds)
        ax.plot(x, means, color=color, marker="o", label=group_name)
        ax.fill_between(x, means - stds, means + stds, color=color, alpha=0.16)
    style_ax(ax, "Perfil medio y variabilidad de los coeficientes MFCC por clase en sireNNet", "Coeficiente MFCC", "Valor del coeficiente MFCC")
    ax.set_xticks(list(x))
    ax.legend(title="Carpeta", frameon=False)
    fig.text(0.01, -0.03, "La línea es la media de los MFCC medios por archivo. La banda es ±1 desviación estándar en la muestra. No indica capacidad predictiva.", fontsize=8)
    fig.tight_layout()
    save_figure(fig, "Figura_3_8_MFCC_sireNNet.png")


def section_a3_figure() -> None:
    features = STATE["features"]
    a3 = features[(features["dataset_id"] == "a3siren") & (features["feature_error"].fillna("") == "")].copy()
    generator = np.random.default_rng(SEED)
    picks = []
    for group_name in ("Siren", "Noise"):
        ids = sorted(a3.loc[a3["analysis_group"] == group_name, "track_id"].dropna().unique())
        if not ids:
            continue
        chosen = list(generator.choice(ids, size=min(2, len(ids)), replace=False))
        picks.extend((group_name, track_id) for track_id in sorted(chosen))
    STATE["a3_example_tracks"] = picks
    fig, axes = plt.subplots(2, 2, figsize=(11, 7.4))
    axes = list(axes.ravel())
    for ax in axes:
        ax.set_axis_off()
    for ax, (group_name, track_id) in zip(axes, picks):
        ax.set_axis_on()
        subset = a3[a3["track_id"] == track_id].copy()
        subset["channel"] = subset.apply(lambda row: a3_channel(row.relative_path, Path(row.relative_path).name), axis=1)
        subset["channel_index"] = subset["channel"].str.replace("ch", "", regex=False).astype(int)
        subset = subset.sort_values("channel_index")
        ax.plot(subset["channel"], subset["rms_mean"], marker="o", color="#4C72B0")
        style_ax(ax, f"{group_name}: {track_id}", "Canal", "RMS (amplitud)")
    fig.suptitle("Variabilidad de RMS entre canales de una misma toma en A3Siren-Recordings", fontsize=13)
    fig.text(0.01, -0.02, "Cada conjunto de ocho canales corresponde a una misma toma acústica. Los canales no son observaciones independientes.", fontsize=8)
    fig.tight_layout()
    save_figure(fig, "Figura_3_8_Variabilidad_A3Siren.png")

section_mfcc()


## 08. Canales de A3Siren-Recordings

Cada panel es una toma. Los ocho canales no se tratan como ocho observaciones independientes.


In [ ]:
def section_a3_figure() -> None:
    features = STATE["features"]
    a3 = features[(features["dataset_id"] == "a3siren") & (features["feature_error"].fillna("") == "")].copy()
    generator = np.random.default_rng(SEED)
    picks = []
    for group_name in ("Siren", "Noise"):
        ids = sorted(a3.loc[a3["analysis_group"] == group_name, "track_id"].dropna().unique())
        if not ids:
            continue
        chosen = list(generator.choice(ids, size=min(2, len(ids)), replace=False))
        picks.extend((group_name, track_id) for track_id in sorted(chosen))
    STATE["a3_example_tracks"] = picks
    fig, axes = plt.subplots(2, 2, figsize=(11, 7.4))
    axes = list(axes.ravel())
    for ax in axes:
        ax.set_axis_off()
    for ax, (group_name, track_id) in zip(axes, picks):
        ax.set_axis_on()
        subset = a3[a3["track_id"] == track_id].copy()
        subset["channel"] = subset.apply(lambda row: a3_channel(row.relative_path, Path(row.relative_path).name), axis=1)
        subset["channel_index"] = subset["channel"].str.replace("ch", "", regex=False).astype(int)
        subset = subset.sort_values("channel_index")
        ax.plot(subset["channel"], subset["rms_mean"], marker="o", color="#4C72B0")
        style_ax(ax, f"{group_name}: {track_id}", "Canal", "RMS (amplitud)")
    fig.suptitle("Variabilidad de RMS entre canales de una misma toma en A3Siren-Recordings", fontsize=13)
    fig.text(0.01, -0.02, "Cada conjunto de ocho canales corresponde a una misma toma acústica. Los canales no son observaciones independientes.", fontsize=8)
    fig.tight_layout()
    save_figure(fig, "Figura_3_8_Variabilidad_A3Siren.png")

section_a3_figure()


## 09. Contraste descriptivo dentro de AudioSet-EV v2 y de LSSiren

Las etiquetas de un corpus no se mezclan con las del otro. RMS y centroide se muestran por separado.


In [ ]:
def section_siren_noise() -> None:
    features = STATE["features"]
    fig, axes = plt.subplots(2, 2, figsize=(11, 8))
    plan = (
        (0, "audioset_ev", "rms_mean", "RMS (amplitud)"),
        (1, "audioset_ev", "spectral_centroid_mean", "Centroide espectral (Hz)"),
        (2, "lssiren", "rms_mean", "RMS (amplitud)"),
        (3, "lssiren", "spectral_centroid_mean", "Centroide espectral (Hz)"),
    )
    for position, dataset_id, column, ylabel in plan:
        ax = axes.ravel()[position]
        subset = features[(features["dataset_id"] == dataset_id) & (features["feature_error"].fillna("") == "")]
        groups = ordered_groups(subset, dataset_id)
        draw_box(ax, subset, column, groups)
        style_ax(ax, DISPLAY[dataset_id], "Grupo observado en este corpus", ylabel)
    fig.suptitle("Energía y centroide espectral en AudioSet-EV v2 y LSSiren", fontsize=13)
    fig.text(0.01, -0.02, "Cada fila es un corpus distinto. Las etiquetas no se combinan entre datasets. La caja muestra solapamiento, no una regla de clasificación.", fontsize=8)
    fig.tight_layout()
    save_figure(fig, "Figura_3_5_Sirena_Ruido.png")

section_siren_noise()


## 10. Centroide entre datasets

La mediana y el intervalo intercuartílico resumen la muestra. A3Siren-Recordings entra como una mediana por track. La diferencia entre corpus no se lee como domain shift.


In [ ]:
def section_dataset_comparison() -> None:
    features = STATE["features"]
    comparable = features[features["dataset_id"] != "a3siren"].copy()
    a3 = features[features["dataset_id"] == "a3siren"]
    if not a3.empty:
        track = a3.groupby(["track_id", "analysis_group"], as_index=False)[["spectral_centroid_mean"]].median()
        track["dataset"] = DISPLAY["a3siren"]
        track["dataset_id"] = "a3siren"
        comparable = pd.concat([comparable, track], ignore_index=True, sort=False)
    rows = []
    labels = []
    medians = []
    lows = []
    highs = []
    for dataset_id in DATASET_ORDER:
        subset = comparable[comparable["dataset_id"] == dataset_id]
        values = pd.to_numeric(subset["spectral_centroid_mean"], errors="coerce").dropna()
        if values.empty:
            continue
        labels.append(DISPLAY[dataset_id] + ("\n(mediana por track)" if dataset_id == "a3siren" else ""))
        medians.append(float(values.median()))
        lows.append(float(values.quantile(0.25)))
        highs.append(float(values.quantile(0.75)))
        rows.append({"dataset": DISPLAY[dataset_id], "n": int(len(values)), "centroid_median_hz": medians[-1], "centroid_p25_hz": lows[-1], "centroid_p75_hz": highs[-1], "unit": "track" if dataset_id == "a3siren" else "archivo de la muestra"})
    STATE["centroid_comparison"] = pd.DataFrame(rows)
    save_table(STATE["centroid_comparison"], "centroid_comparison_refined.csv")
    fig, ax = plt.subplots(figsize=(8.8, 5.2))
    x = np.arange(len(labels))
    ax.vlines(x, lows, highs, color="#4C72B0", linewidth=4)
    ax.scatter(x, medians, color="#8C2F39", zorder=3)
    ax.set_xticks(x, labels)
    style_ax(ax, "Mediana e intervalo intercuartílico del centroide espectral", "Dataset", "Centroide espectral (Hz)")
    fig.text(0.01, -0.08, "Las diferencias entre corpus se interpretan como heterogeneidad descriptiva y no constituyen por sí mismas una demostración de domain shift.", fontsize=8)
    fig.tight_layout()
    save_figure(fig, "Figura_3_9_Comparacion_Datasets_Refinada.png")

section_dataset_comparison()


## 11. Ejemplos tiempo-frecuencia

Cuatro archivos de la muestra, uno por grupo. Cada panel es un ejemplo.


In [ ]:
def section_spectrogram() -> None:
    features = STATE["features"]
    generator = np.random.default_rng(SEED)
    requests = (
        ("audioset_ev", "Positive_files"),
        ("audioset_ev", "Negative_files"),
        ("lssiren", "Ambulance"),
        ("lssiren", "Road"),
    )
    fig, axes = plt.subplots(2, 2, figsize=(11, 7.6))
    chosen = []
    for ax, (dataset_id, group_name) in zip(axes.ravel(), requests):
        subset = features[(features["dataset_id"] == dataset_id) & (features["analysis_group"] == group_name) & (features["feature_error"].fillna("") == "")].sort_values("relative_path")
        if subset.empty:
            ax.set_axis_off()
            continue
        pick = subset.iloc[int(generator.integers(0, len(subset)))]
        chosen.append({"dataset": pick.dataset, "analysis_group": pick.analysis_group, "relative_path": pick.relative_path})
        waveform, sr, _mix = load_audio(Path(pick.absolute_path))
        spectrum = np.abs(librosa.stft(waveform, n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH, window=WINDOW))
        image = librosa.display.specshow(librosa.amplitude_to_db(spectrum, ref=np.max), sr=sr, hop_length=HOP_LENGTH, x_axis="time", y_axis="log", ax=ax)
        style_ax(ax, f"{pick.dataset}: {pick.analysis_group}", "Tiempo (s)", "Frecuencia (Hz)")
        fig.colorbar(image, ax=ax, format="%+2.0f dB", fraction=0.046)
    STATE["spectrogram_examples"] = pd.DataFrame(chosen)
    fig.suptitle("Ejemplos de representación tiempo-frecuencia", fontsize=13)
    fig.text(0.01, -0.02, "Cada panel es un archivo de la muestra regenerada. No representa a toda la clase ni es una entrada de modelo.", fontsize=8)
    fig.tight_layout()
    save_figure(fig, "Figura_3_6_Espectrograma.png")

section_spectrogram()


## 12. Tablas consolidadas

La duración sale de la población analítica. Los descriptores acústicos salen de la muestra. Esa diferencia queda escrita en las columnas de base. La observación de sireNNet se calcula a partir de las medianas y del solapamiento intercuartílico.


In [ ]:
def section_tables() -> None:
    features = STATE["features"]
    usable = features[features["feature_error"].fillna("") == ""].copy()
    a3 = usable[usable["dataset_id"] == "a3siren"]
    file_level = usable[usable["dataset_id"] != "a3siren"]
    if not a3.empty:
        acoustic_columns = ["rms_mean", "zcr_mean", "dominant_frequency", "spectral_centroid_mean", "spectral_bandwidth_mean", "spectral_rolloff_mean", "spectral_flux_mean", "spectral_flatness_mean"] + [f"mfcc_{index}_mean" for index in range(1, 14)]
        track = a3.groupby(["dataset_id", "dataset", "track_id", "analysis_group"], as_index=False)[acoustic_columns].median()
        track["logical_unit"] = "TRACK"
        file_level = pd.concat([file_level, track], ignore_index=True, sort=False)
    rows = []
    for (dataset_id, group), subset in file_level.groupby(["dataset_id", "analysis_group"], sort=False):
        n_duration, duration_mean, duration_median = _population_duration_stats(dataset_id, group)
        record = {
            "dataset": DISPLAY[dataset_id],
            "grupo": group,
            "n_unidades_duracion": n_duration,
            "duracion_media_s": duration_mean,
            "duracion_mediana_s": duration_median,
            "n_muestra_acustica": int(subset["rms_mean"].notna().sum()) if dataset_id != "a3siren" else int(subset["track_id"].nunique()) if "track_id" in subset.columns else int(len(subset)),
            "rms_media": float(pd.to_numeric(subset["rms_mean"], errors="coerce").mean()),
            "rms_mediana": float(pd.to_numeric(subset["rms_mean"], errors="coerce").median()),
            "zcr_media": float(pd.to_numeric(subset["zcr_mean"], errors="coerce").mean()),
            "zcr_mediana": float(pd.to_numeric(subset["zcr_mean"], errors="coerce").median()),
            "frecuencia_dominante_media_hz": float(pd.to_numeric(subset["dominant_frequency"], errors="coerce").mean()),
            "centroid_media_hz": float(pd.to_numeric(subset["spectral_centroid_mean"], errors="coerce").mean()),
            "bandwidth_media_hz": float(pd.to_numeric(subset["spectral_bandwidth_mean"], errors="coerce").mean()),
            "rolloff_media_hz": float(pd.to_numeric(subset["spectral_rolloff_mean"], errors="coerce").mean()),
            "flux_media": float(pd.to_numeric(subset["spectral_flux_mean"], errors="coerce").mean()),
            "flatness_media": float(pd.to_numeric(subset["spectral_flatness_mean"], errors="coerce").mean()),
            "base_duracion": "población analítica: un archivo por hash, o un track en A3Siren-Recordings",
            "base_acustica": "muestra regenerada con seed 42; en A3Siren-Recordings es la mediana explícita por track",
        }
        for index in range(1, 14):
            record[f"mfcc_{index}_media"] = float(pd.to_numeric(subset[f"mfcc_{index}_mean"], errors="coerce").mean())
        rows.append(record)
    master = pd.DataFrame(rows)
    master["_order"] = master["dataset"].map({DISPLAY[key]: index for index, key in enumerate(DATASET_ORDER)})
    master = master.sort_values(["_order", "grupo"]).drop(columns="_order").reset_index(drop=True)
    STATE["master"] = master
    save_table(master, "Tabla_3_5_Caracteristicas_Acusticas.csv")
    if HAS_OPENPYXL:
        excel_path = OUTPUT_ROOT / "tables" / "Tabla_3_5_Caracteristicas_Acusticas.xlsx"
        master.to_excel(excel_path, index=False)
        print(f"escrito: {excel_path}")
    else:
        print("openpyxl no está disponible. No se escribió el XLSX.")
    inventory = STATE["inventory"]
    summary_rows = []
    for dataset_id in DATASET_ORDER:
        subset = inventory[inventory["dataset_id"] == dataset_id]
        valid = subset[subset["quality_status"] == "OK"]
        units = STATE["duration_units"]
        unit_subset = units[units["dataset_id"] == dataset_id]
        rates = sorted(pd.to_numeric(valid["sampling_rate_hz"], errors="coerce").dropna().astype(int).unique())
        channels = sorted(pd.to_numeric(valid["channels"], errors="coerce").dropna().astype(int).unique())
        summary_rows.append({
            "dataset": DISPLAY[dataset_id],
            "archivos": int(len(subset)),
            "utilizables": int(len(valid)),
            "invalidos": int((subset["quality_status"] != "OK").sum()),
            "duplicados_secundarios": int((~valid["is_hash_primary"]).sum()) if dataset_id != "a3siren" else 0,
            "unidad_analitica": "TRACK" if dataset_id == "a3siren" else "FILE",
            "n_unidades": int(len(unit_subset)),
            "grupos": " | ".join(ordered_groups(unit_subset, dataset_id)),
            "tasas_hz": " | ".join(str(value) for value in rates),
            "canales": " | ".join(str(value) for value in channels),
        })
    summary = pd.DataFrame(summary_rows)
    STATE["summary"] = summary
    save_table(summary, "Tabla_3_1_Resumen_Datasets.csv")
    save_table(master[["dataset", "grupo", "n_unidades_duracion", "duracion_media_s", "duracion_mediana_s", "zcr_media", "zcr_mediana", "base_duracion", "base_acustica"]], "Tabla_3_2_Caracteristicas_Temporales.csv")
    save_table(master[["dataset", "grupo", "n_muestra_acustica", "rms_media", "rms_mediana", "base_acustica"]], "Tabla_3_3_Caracteristicas_Energeticas.csv")
    save_table(master[["dataset", "grupo", "n_muestra_acustica", "frecuencia_dominante_media_hz", "centroid_media_hz", "bandwidth_media_hz", "rolloff_media_hz", "flux_media", "flatness_media", "base_acustica"]], "Tabla_3_4_Caracteristicas_Frecuenciales.csv")
    siren_units = STATE["duration_units"]
    siren_features = features[(features["dataset_id"] == "sirennet") & (features["feature_error"].fillna("") == "")]
    siren_rows = []
    for group in SIREN_FOLDERS:
        subset = siren_features[siren_features["analysis_group"] == group]
        if subset.empty:
            continue
        duration_values = pd.to_numeric(siren_units.loc[(siren_units["dataset_id"] == "sirennet") & (siren_units["analysis_group"] == group), "duration_seconds"], errors="coerce").dropna()
        others = siren_features[siren_features["analysis_group"] != group]
        rms_median = float(pd.to_numeric(subset["rms_mean"], errors="coerce").median())
        phrases = []
        if group != "traffic" and (others["analysis_group"] == "traffic").any():
            traffic_median = float(pd.to_numeric(others.loc[others["analysis_group"] == "traffic", "rms_mean"], errors="coerce").median())
            if rms_median > traffic_median:
                phrases.append(f"La mediana de RMS ({rms_median:.4f}) es mayor que la de traffic ({traffic_median:.4f})")
            elif rms_median < traffic_median:
                phrases.append(f"La mediana de RMS ({rms_median:.4f}) es menor que la de traffic ({traffic_median:.4f})")
            else:
                phrases.append(f"La mediana de RMS ({rms_median:.4f}) coincide con la de traffic")
        overlapped = []
        for other, other_frame in others.groupby("analysis_group"):
            _ratio, phrase = iqr_overlap(subset["rms_mean"], other_frame["rms_mean"])
            if "solapamiento" in phrase or "parcialmente" in phrase:
                overlapped.append(str(other))
        if overlapped:
            phrases.append("el rango intercuartílico de RMS se solapa con " + ", ".join(overlapped))
        siren_rows.append({
            "grupo": group,
            "n": int(len(subset)),
            "duration_median": float(duration_values.median()) if not duration_values.empty else float("nan"),
            "rms_median": rms_median,
            "zcr_median": float(pd.to_numeric(subset["zcr_mean"], errors="coerce").median()),
            "dominant_frequency_median": float(pd.to_numeric(subset["dominant_frequency"], errors="coerce").median()),
            "centroid_median": float(pd.to_numeric(subset["spectral_centroid_mean"], errors="coerce").median()),
            "bandwidth_median": float(pd.to_numeric(subset["spectral_bandwidth_mean"], errors="coerce").median()),
            "rolloff_median": float(pd.to_numeric(subset["spectral_rolloff_mean"], errors="coerce").median()),
            "flux_median": float(pd.to_numeric(subset["spectral_flux_mean"], errors="coerce").median()),
            "observacion_descriptiva": _sentence(phrases),
        })
    siren_table = pd.DataFrame(siren_rows)
    STATE["sirennet_table"] = siren_table
    save_table(siren_table, "Tabla_3_6_sireNNet.csv")
    show_df(siren_table, n=10)

section_tables()


## 13. Cinco hallazgos y puente al Capítulo 4

Hay exactamente cinco hallazgos. Cada implicación es una acción posible para la preparación. Ninguna se ejecuta aquí.


In [ ]:
def section_findings() -> None:
    inventory = STATE["inventory"]
    units = STATE["duration_units"]
    features = STATE["features"]
    usable = features[features["feature_error"].fillna("") == ""].copy()
    summary = STATE["summary"]
    rate_text = []
    channel_text = []
    duration_text = []
    for record in summary.itertuples(index=False):
        rate_text.append(f"{record.dataset}: {record.tasas_hz} Hz")
        channel_text.append(f"{record.dataset}: {record.canales} canal(es)")
        subset = pd.to_numeric(units.loc[units["dataset"] == record.dataset, "duration_seconds"], errors="coerce").dropna()
        if not subset.empty:
            duration_text.append(f"{record.dataset}: mediana {_fmt(float(subset.median()))} s, IQR {_fmt(float(subset.quantile(0.75) - subset.quantile(0.25)))} s, n={len(subset)} {record.unidad_analitica}")
    pair_notes = []
    pair_plan = (
        ("audioset_ev", "Positive_files", "Negative_files", "rms_mean"),
        ("lssiren", "Ambulance", "Road", "rms_mean"),
        ("lssiren", "Ambulance", "Road", "spectral_centroid_mean"),
    )
    for dataset_id, left, right, column in pair_plan:
        subset = usable[usable["dataset_id"] == dataset_id]
        ratio, phrase = iqr_overlap(subset.loc[subset["analysis_group"] == left, column], subset.loc[subset["analysis_group"] == right, column])
        left_median = pd.to_numeric(subset.loc[subset["analysis_group"] == left, column], errors="coerce").median()
        right_median = pd.to_numeric(subset.loc[subset["analysis_group"] == right, column], errors="coerce").median()
        pair_notes.append(f"{DISPLAY[dataset_id]} {column}: mediana {left}={_fmt(float(left_median), 4)}, mediana {right}={_fmt(float(right_median), 4)}, solapamiento IQR={_fmt(ratio)}; {phrase}")
    siren = usable[usable["dataset_id"] == "sirennet"]
    siren_notes = []
    classes = ordered_groups(siren, "sirennet")
    for index, left in enumerate(classes):
        for right in classes[index + 1:]:
            ratio, phrase = iqr_overlap(siren.loc[siren["analysis_group"] == left, "spectral_centroid_mean"], siren.loc[siren["analysis_group"] == right, "spectral_centroid_mean"])
            siren_notes.append(f"{left} frente a {right}: solapamiento IQR del centroide={_fmt(ratio)}; {phrase}")
    a3 = usable[usable["dataset_id"] == "a3siren"]
    if not a3.empty:
        track_rms = a3.groupby(["track_id", "analysis_group"], as_index=False)["rms_mean"].median()
        ratio, phrase = iqr_overlap(track_rms.loc[track_rms["analysis_group"] == "Siren", "rms_mean"], track_rms.loc[track_rms["analysis_group"] == "Noise", "rms_mean"])
        pair_notes.append(f"A3Siren-Recordings RMS por track: solapamiento IQR={_fmt(ratio)}; {phrase}")
        channel_range = a3.groupby("track_id")["rms_mean"].agg(lambda series: float(series.max() - series.min()))
        range_median = float(channel_range.median()) if not channel_range.empty else float("nan")
    else:
        range_median = float("nan")
    within_std = []
    dataset_medians = []
    for dataset_id, subset in usable.groupby("dataset_id"):
        if dataset_id == "a3siren":
            subset = subset.groupby(["track_id", "analysis_group"], as_index=False)["spectral_centroid_mean"].median()
        values = pd.to_numeric(subset["spectral_centroid_mean"], errors="coerce").dropna()
        if len(values) >= 2:
            within_std.append(float(values.std(ddof=1)))
        if not values.empty:
            dataset_medians.append(float(values.median()))
    within_mean = float(np.mean(within_std)) if within_std else float("nan")
    between_span = float(max(dataset_medians) - min(dataset_medians)) if len(dataset_medians) >= 2 else float("nan")
    lssiren = inventory[(inventory["dataset_id"] == "lssiren") & (inventory["quality_status"] == "OK")]
    lssiren_rates = sorted(pd.to_numeric(lssiren["sampling_rate_hz"], errors="coerce").dropna().astype(int).unique())
    mono = int((pd.to_numeric(lssiren["channels"], errors="coerce") == 1).sum())
    stereo = int((pd.to_numeric(lssiren["channels"], errors="coerce") == 2).sum())
    findings = pd.DataFrame([
        {
            "finding_id": "H1",
            "dataset": "todos",
            "observation": "Los cuatro corpus difieren en frecuencia de muestreo, número de canales y duración de sus unidades de análisis.",
            "quantitative_evidence": "Tasas: " + "; ".join(rate_text) + ". Canales: " + "; ".join(channel_text) + ".",
            "interpretation": "Se observa heterogeneidad estructural entre corpus. Esa diferencia describe cómo fueron construidos los datos.",
            "implication_for_preparation": "Durante la preparación deberá evaluarse si hace falta armonizar tasa, canales o duración. Esta notebook no aplica esa armonización.",
            "confidence": "HIGH",
        },
        {
            "finding_id": "H2",
            "dataset": "todos",
            "observation": "La duración de las unidades de análisis no comparte la misma mediana ni el mismo rango intercuartílico entre corpus.",
            "quantitative_evidence": "; ".join(duration_text) + ".",
            "interpretation": "Se identifican escalas temporales distintas. AudioSet-EV v2, LSSiren, sireNNet y los tracks de A3Siren-Recordings no quedan descritos por una única duración típica.",
            "implication_for_preparation": "Durante la preparación deberá evaluarse una estrategia de segmentación o ventaneo. Esta notebook no segmenta los archivos.",
            "confidence": "HIGH",
        },
        {
            "finding_id": "H3",
            "dataset": "todos, con etiquetas propias de cada corpus",
            "observation": "Algunos grupos observados difieren en RMS o en centroide espectral, y sus rangos intercuartílicos se solapan.",
            "quantitative_evidence": " ".join(pair_notes + siren_notes) + ".",
            "interpretation": "Se observan diferencias descriptivas junto con solapamiento. Una variable aislada no queda caracterizada como una regla de clasificación.",
            "implication_for_preparation": "Durante la preparación deberá evaluarse una representación multivariable. Esta notebook no selecciona características definitivas.",
            "confidence": "MEDIUM",
        },
        {
            "finding_id": "H4",
            "dataset": "todos",
            "observation": "El centroide espectral presenta dispersión dentro de cada corpus y sus medianas no coinciden entre corpus.",
            "quantitative_evidence": f"Desviación estándar media del centroide dentro de cada corpus, tras resumir A3Siren-Recordings por track: {_fmt(within_mean)} Hz. Diferencia entre la mediana más alta y la más baja de los cuatro corpus: {_fmt(between_span)} Hz.",
            "interpretation": "Se identifica variabilidad intraclase y variación entre corpus. La variación entre corpus se interpreta como heterogeneidad descriptiva, no como domain shift demostrado.",
            "implication_for_preparation": "Durante la preparación deberá conservarse la identidad de cada corpus al comparar o combinar representaciones.",
            "confidence": "MEDIUM",
        },
        {
            "finding_id": "H5",
            "dataset": "LSSiren | A3Siren-Recordings",
            "observation": "LSSiren mezcla condiciones técnicas de captura y A3Siren-Recordings reparte cada toma en ocho canales dependientes.",
            "quantitative_evidence": f"LSSiren: {len(lssiren_rates)} tasas ({lssiren_rates}), {mono} archivos mono y {stereo} archivos estéreo. A3Siren-Recordings: mediana del rango de RMS entre canales de un mismo track = {_fmt(range_median, 4)}.",
            "interpretation": "La variación entre canales pertenece a una misma toma. Las distintas tasas y configuraciones de canal de LSSiren son heterogeneidad interna del corpus.",
            "implication_for_preparation": "Una partición futura de A3Siren-Recordings debe usar el track como unidad. En LSSiren deberá evaluarse el tratamiento de canales y de sampling rate.",
            "confidence": "HIGH",
        },
    ])
    if len(findings) != 5:
        raise AssertionError("La tabla de hallazgos debe contener exactamente cinco filas.")
    STATE["findings"] = findings
    save_table(findings, "Tabla_3_7_Hallazgos_Iniciales.csv")
    show_df(findings[["finding_id", "dataset", "confidence"]], n=5)
    secondary = int(((inventory["quality_status"] == "OK") & (~inventory["is_hash_primary"]) & (inventory["dataset_id"] != "a3siren")).sum())
    bridge = [
        {"finding": "Frecuencia de muestreo heterogénea", "evidence": "; ".join(rate_text), "possible_action": "evaluar armonización de sampling rate", "reason": "Los corpus no comparten el mismo Nyquist.", "section_chapter4": "Preparación: armonización de tasa"},
        {"finding": "Canales mono y estéreo en LSSiren", "evidence": f"{mono} mono y {stereo} estéreo.", "possible_action": "evaluar tratamiento de canales", "reason": "La mezcla en memoria de esta notebook no queda guardada.", "section_chapter4": "Preparación: canales"},
        {"finding": "Duraciones distintas", "evidence": "; ".join(duration_text), "possible_action": "evaluar segmentación o ventaneo", "reason": "Las unidades de análisis no comparten la misma escala temporal.", "section_chapter4": "Preparación: segmentación"},
        {"finding": "Copias exactas secundarias", "evidence": f"{secondary} archivos utilizables marcados como copia secundaria de un hash ya conservado.", "possible_action": "evitar separar copias de un mismo hash entre particiones", "reason": "Una copia exacta no es una observación nueva.", "section_chapter4": "Preparación: partición"},
        {"finding": "A3Siren track por ocho canales", "evidence": f"Mediana del rango de RMS entre canales de un track = {_fmt(range_median, 4)}.", "possible_action": "partir por track y conservar los canales sin tratarlos como muestras independientes", "reason": "Los canales de una toma están estructuralmente dependientes.", "section_chapter4": "Preparación: unidad track"},
        {"finding": "Diferencias acústicas con solapamiento", "evidence": " ".join(pair_notes[:2]), "possible_action": "evaluar una representación multivariable", "reason": "El solapamiento de una sola variable no sostiene una regla de clasificación.", "section_chapter4": "Preparación: representación"},
    ]
    bridge_frame = pd.DataFrame(bridge)
    STATE["bridge"] = bridge_frame
    save_table(bridge_frame, "Hallazgos_Puente_Capitulo_4.csv")

section_findings()


## 14. Selección para el capítulo

PRINCIPAL indica una figura o tabla útil para explicar el capítulo 3. SECUNDARIA aporta detalle. ANEXO ilustra sin resumir la población.


In [ ]:
def section_selection() -> None:
    figures = pd.DataFrame([
        {"figure_id": "Figura 3.1", "file": "plots/Figura_3_1_Unidades_Analisis.png", "title": "Distribución de las unidades de análisis por dataset y grupo observado", "section": "3.3 Exploración de los Datos", "purpose": "Mostrar cuántas unidades analíticas hay en cada grupo observado.", "evidence": "Conteos regenerados de archivos primarios y de tracks.", "why_it_is_useful": "Fija la unidad antes de comparar duraciones o características.", "recommended_use": "Cuerpo del capítulo, junto con la tabla de resumen.", "status": "PRINCIPAL"},
        {"figure_id": "Figura 3.2", "file": "plots/Figura_3_2_Duracion.png", "title": "Distribución de la duración de las unidades de análisis por dataset", "section": "3.3 Exploración de los Datos", "purpose": "Comparar la duración sin mezclar archivos y tracks.", "evidence": "Mediana e IQR de la población analítica.", "why_it_is_useful": "Sostiene el hallazgo de escalas temporales distintas.", "recommended_use": "Cuerpo del capítulo. Las alternativas de un solo panel quedan como apoyo.", "status": "PRINCIPAL"},
        {"figure_id": "Figura 3.3", "file": "plots/Figura_3_3_Sampling_Rate.png", "title": "Frecuencias de muestreo observadas en los archivos de cada corpus", "section": "3.3 Exploración de los Datos", "purpose": "Mostrar la heterogeneidad técnica de tasa.", "evidence": "Conteo de archivos por sampling rate.", "why_it_is_useful": "Justifica evaluar una armonización sin ejecutarla.", "recommended_use": "Cuerpo del capítulo.", "status": "PRINCIPAL"},
        {"figure_id": "Figura 3.4", "file": "plots/Figura_3_4_RMS_sireNNet.png", "title": "RMS por carpeta en sireNNet", "section": "3.4 Análisis de Variables Relevantes", "purpose": "Describir la energía de las cuatro carpetas.", "evidence": "Boxplot de RMS en la muestra acústica.", "why_it_is_useful": "Detalla un corpus, aunque la comparación sirena/ruido ya usa RMS en otro corpus.", "recommended_use": "Texto de 3.4 o anexo si el espacio del capítulo es limitado.", "status": "SECUNDARIA"},
        {"figure_id": "Figura 3.5", "file": "plots/Figura_3_5_Sirena_Ruido.png", "title": "Energía y centroide espectral en AudioSet-EV v2 y LSSiren", "section": "3.4 Análisis de Variables Relevantes", "purpose": "Mostrar diferencias y solapamiento sin fusionar corpus.", "evidence": "Boxplots de RMS y centroide por grupo observado.", "why_it_is_useful": "Sostiene el hallazgo de diferencias acústicas con solapamiento.", "recommended_use": "Cuerpo del capítulo.", "status": "PRINCIPAL"},
        {"figure_id": "Figura 3.6", "file": "plots/Figura_3_6_Espectrograma.png", "title": "Ejemplos de representación tiempo-frecuencia", "section": "3.3 Exploración de los Datos", "purpose": "Ilustrar la forma tiempo-frecuencia de cuatro archivos.", "evidence": "Cuatro ejemplos reproducibles de la muestra.", "why_it_is_useful": "Ayuda a leer las señales, pero no resume la población.", "recommended_use": "Anexo, con la aclaración de que cada panel es un ejemplo.", "status": "ANEXO"},
        {"figure_id": "Figura 3.7", "file": "plots/Figura_3_7_Caracteristicas_Espectrales_sireNNet.png", "title": "Distribución de características espectrales por clase en sireNNet", "section": "3.4 Análisis de Variables Relevantes", "purpose": "Comparar centroide, ancho de banda, roll-off y frecuencia dominante.", "evidence": "Cuatro boxplots con escalas independientes.", "why_it_is_useful": "Muestra desplazamiento, dispersión y solapamiento dentro de un mismo corpus.", "recommended_use": "Cuerpo de 3.4.", "status": "PRINCIPAL"},
        {"figure_id": "Figura 3.8", "file": "plots/Figura_3_8_MFCC_sireNNet.png", "title": "Perfil medio y variabilidad de los coeficientes MFCC por clase en sireNNet", "section": "3.4 Análisis de Variables Relevantes", "purpose": "Mostrar la media y la dispersión de 13 MFCC.", "evidence": "Media y banda de ±1 desviación estándar por carpeta.", "why_it_is_useful": "Describe variabilidad cepstral sin convertir un coeficiente en predictor.", "recommended_use": "Cuerpo de 3.4 si hay espacio; si no, anexo.", "status": "SECUNDARIA"},
        {"figure_id": "Figura 3.9", "file": "plots/Figura_3_8_Variabilidad_A3Siren.png", "title": "Variabilidad de RMS entre canales de una misma toma en A3Siren-Recordings", "section": "3.3 Exploración de los Datos", "purpose": "Mostrar que los canales de una toma no son idénticos ni independientes.", "evidence": "RMS de ch1 a ch8 en dos tracks Siren y dos tracks Noise, o en todos los tracks disponibles si hay menos.", "why_it_is_useful": "Sostiene la unidad track para el capítulo de preparación.", "recommended_use": "Cuerpo del capítulo. El nombre del archivo conserva el identificador pedido en la tarea de A3Siren.", "status": "PRINCIPAL"},
        {"figure_id": "Figura 3.10", "file": "plots/Figura_3_9_Comparacion_Datasets_Refinada.png", "title": "Mediana e intervalo intercuartílico del centroide espectral", "section": "3.3 Exploración de los Datos", "purpose": "Resumir el centroide por corpus con la nota de heterogeneidad.", "evidence": "Mediana y percentiles 25 y 75. A3Siren-Recordings entra como mediana por track.", "why_it_is_useful": "Evita leer la diferencia entre corpus como domain shift confirmado.", "recommended_use": "Anexo o apoyo de 3.3, no como prueba de cambio de dominio.", "status": "SECUNDARIA"},
    ])
    tables = pd.DataFrame([
        {"table_id": "Tabla 3.1", "file": "tables/Tabla_3_1_Resumen_Datasets.csv", "title": "Resumen de datasets", "section": "3.3 Exploración de los Datos", "purpose": "Fijar archivos, unidades, tasas y canales.", "status": "PRINCIPAL", "notes": "Regenerada desde cabeceras y hashes."},
        {"table_id": "Tabla 3.2", "file": "tables/Tabla_3_2_Caracteristicas_Temporales.csv", "title": "Características temporales", "section": "3.4 Análisis de Variables Relevantes", "purpose": "Duración de la población y ZCR de la muestra.", "status": "SECUNDARIA", "notes": "La duración y el ZCR no salen de la misma base; las columnas de base lo indican."},
        {"table_id": "Tabla 3.3", "file": "tables/Tabla_3_3_Caracteristicas_Energeticas.csv", "title": "Características energéticas", "section": "3.4 Análisis de Variables Relevantes", "purpose": "RMS medio y mediano por grupo.", "status": "SECUNDARIA", "notes": "Calculada sobre la muestra acústica; A3Siren-Recordings está resumido por track."},
        {"table_id": "Tabla 3.4", "file": "tables/Tabla_3_4_Caracteristicas_Frecuenciales.csv", "title": "Características frecuenciales", "section": "3.4 Análisis de Variables Relevantes", "purpose": "Medias espectrales por grupo.", "status": "SECUNDARIA", "notes": "Un valor en hertz no es equivalente entre tasas distintas de LSSiren."},
        {"table_id": "Tabla 3.5", "file": "tables/Tabla_3_5_Caracteristicas_Acusticas.csv", "title": "Características acústicas consolidadas", "section": "3.4 Análisis de Variables Relevantes", "purpose": "Reunir duración poblacional y descriptores de la muestra.", "status": "PRINCIPAL", "notes": "También se escribe XLSX cuando openpyxl está disponible."},
        {"table_id": "Tabla 3.6", "file": "tables/Tabla_3_6_sireNNet.csv", "title": "Características acústicas descriptivas por clase en sireNNet", "section": "3.4 Análisis de Variables Relevantes", "purpose": "Medianas por carpeta y una observación calculada.", "status": "PRINCIPAL", "notes": "La observación solo compara medianas y solapamiento de RMS medidos en la muestra."},
        {"table_id": "Tabla 3.7", "file": "tables/Tabla_3_7_Hallazgos_Iniciales.csv", "title": "Hallazgos iniciales", "section": "3.5 Hallazgos Iniciales", "purpose": "Cinco hallazgos con evidencia, interpretación e implicación.", "status": "PRINCIPAL", "notes": "La confianza es descriptiva, no la de un modelo."},
    ])
    STATE["figure_selection"] = figures
    STATE["table_selection"] = tables
    save_table(figures, "thesis_figure_selection_final.csv")
    save_table(tables, "thesis_table_selection_final.csv")
    print("Figuras principales:", ", ".join(figures.loc[figures["status"] == "PRINCIPAL", "figure_id"]))
    print("Figuras secundarias:", ", ".join(figures.loc[figures["status"] == "SECUNDARIA", "figure_id"]))
    print("Figuras de anexo:", ", ".join(figures.loc[figures["status"] == "ANEXO", "figure_id"]))

section_selection()


## 15. Metadata de reproducibilidad

Cada figura y cada tabla queda ligada a su muestra, su semilla y su tabla de origen.


In [ ]:
def section_metadata() -> None:
    sample = STATE["sample"]
    units = STATE["duration_units"]
    acoustic_n = int((sample["dataset_id"] != "a3siren").sum())
    a3_tracks = int(sample.loc[sample["dataset_id"] == "a3siren", "track_id"].nunique())
    common = {
        "seed": SEED,
        "parameters": "n_fft=2048; hop=512; window=hann; n_mels=64; n_mfcc=13; fmin=20 Hz; fmax=Nyquist; roll-off=0.85; flujo=norma L2 de espectros de magnitud normalizados por trama",
    }
    figure_rows = [
        {"figure_id": "Figura 3.1", "filename": "Figura_3_1_Unidades_Analisis.png", "dataset": "todos", "sample_size": int(len(units)), "selection_method": "población analítica: archivo primario o track", "features": "conteo por grupo", "source_tables": "refined_duration_units.csv", "interpretation_notes": "A3Siren-Recordings cuenta tracks."},
        {"figure_id": "Figura 3.2", "filename": "Figura_3_2_Duracion.png", "dataset": "todos", "sample_size": int(len(units)), "selection_method": "dos paneles por diferencia de unidad", "features": "duration_seconds", "source_tables": "refined_duration_units.csv", "interpretation_notes": STATE.get("duration_decision", "")},
        {"figure_id": "Figura 3.3", "filename": "Figura_3_3_Sampling_Rate.png", "dataset": "todos", "sample_size": int((STATE["inventory"]["quality_status"] == "OK").sum()), "selection_method": "todos los archivos utilizables", "features": "sampling_rate_hz", "source_tables": "Tabla_3_1_Resumen_Datasets.csv", "interpretation_notes": "El conteo de A3Siren-Recordings es de archivos; varios archivos pertenecen al mismo track."},
        {"figure_id": "Figura 3.4", "filename": "Figura_3_4_RMS_sireNNet.png", "dataset": "sireNNet", "sample_size": int((sample["dataset_id"] == "sirennet").sum()), "selection_method": "muestra por carpeta, seed 42, un archivo por hash", "features": "rms_mean", "source_tables": "refined_acoustic_sample.csv", "interpretation_notes": "No se declara separabilidad."},
        {"figure_id": "Figura 3.5", "filename": "Figura_3_5_Sirena_Ruido.png", "dataset": "AudioSet-EV v2 | LSSiren", "sample_size": int(sample["dataset_id"].isin(["audioset_ev", "lssiren"]).sum()), "selection_method": "muestra regenerada, seed 42", "features": "rms_mean | spectral_centroid_mean", "source_tables": "refined_acoustic_sample.csv", "interpretation_notes": "Los corpus no se fusionan."},
        {"figure_id": "Figura 3.6", "filename": "Figura_3_6_Espectrograma.png", "dataset": "AudioSet-EV v2 | LSSiren", "sample_size": int(len(STATE.get("spectrogram_examples", []))), "selection_method": "un archivo por grupo, ordenado por ruta y elegido con seed 42", "features": "espectrograma de magnitud en dB", "source_tables": "refined_acoustic_sample.csv", "interpretation_notes": "Ejemplos, no prototipos de clase."},
        {"figure_id": "Figura 3.7", "filename": "Figura_3_7_Caracteristicas_Espectrales_sireNNet.png", "dataset": "sireNNet", "sample_size": int((sample["dataset_id"] == "sirennet").sum()), "selection_method": "muestra por carpeta, seed 42", "features": "centroid | bandwidth | rolloff | dominant_frequency", "source_tables": "refined_acoustic_sample.csv", "interpretation_notes": "Escalas independientes por panel."},
        {"figure_id": "Figura 3.8", "filename": "Figura_3_8_MFCC_sireNNet.png", "dataset": "sireNNet", "sample_size": int((sample["dataset_id"] == "sirennet").sum()), "selection_method": "muestra por carpeta, seed 42", "features": "mfcc_1_mean a mfcc_13_mean", "source_tables": "refined_acoustic_sample.csv", "interpretation_notes": "Banda de ±1 desviación estándar. No es una medida de importancia."},
        {"figure_id": "Figura 3.9", "filename": "Figura_3_8_Variabilidad_A3Siren.png", "dataset": "A3Siren-Recordings", "sample_size": a3_tracks, "selection_method": "hasta dos tracks Siren y dos tracks Noise con seed 42; se grafican sus ocho canales", "features": "rms_mean por canal", "source_tables": "refined_acoustic_sample.csv", "interpretation_notes": "Cada panel es una toma."},
        {"figure_id": "Figura 3.10", "filename": "Figura_3_9_Comparacion_Datasets_Refinada.png", "dataset": "todos", "sample_size": acoustic_n + a3_tracks, "selection_method": "muestra acústica; A3Siren-Recordings agregado por track antes de la mediana", "features": "spectral_centroid_mean", "source_tables": "centroid_comparison_refined.csv", "interpretation_notes": "Heterogeneidad descriptiva, no domain shift."},
    ]
    for row in figure_rows:
        row.update(common)
    figure_meta = pd.DataFrame(figure_rows)
    save_table(figure_meta, "figure_metadata.csv")
    table_rows = []
    for record in STATE["table_selection"].itertuples(index=False):
        table_rows.append({
            "table_id": record.table_id,
            "filename": Path(record.file).name,
            "datasets": "sireNNet" if "sireNNet" in record.title else "todos",
            "groups": "carpetas ambulance, firetruck, police y traffic" if "sireNNet" in record.title else "grupos observados dentro de cada corpus",
            "sample_size": "ver columnas n de la tabla",
            "source_csv": "refined_duration_units.csv | refined_acoustic_sample.csv",
            "calculation_method": record.notes,
            "notes": "Regenerada en esta notebook. No proviene de un CSV de la Notebook 01.",
        })
    table_rows.append({
        "table_id": "Puente Capítulo 4",
        "filename": "Hallazgos_Puente_Capitulo_4.csv",
        "datasets": "todos",
        "groups": "no aplica",
        "sample_size": "población analítica y muestra acústica",
        "source_csv": "Tabla_3_7_Hallazgos_Iniciales.csv",
        "calculation_method": "Cada acción se deriva de un conteo o de un contraste ya calculado.",
        "notes": "Ninguna acción queda ejecutada.",
    })
    save_table(pd.DataFrame(table_rows), "table_metadata.csv")

section_metadata()


## 16. Informe y validación

El informe separa evidencia, interpretación e implicación. La validación comprueba unidades, duplicados, semilla y ausencia de escritura en el input.


In [ ]:
def section_report() -> None:
    findings = STATE["findings"]
    figures = STATE["figure_selection"]
    lines = [
        "# Refinamiento de Evidencia para la Tesis",
        "",
        "## 1. Objetivo",
        "",
        "Preparar figuras, tablas y cinco hallazgos para redactar las secciones 3.3, 3.4 y 3.5. Esta notebook no repite la auditoría ni entrena modelos. Los descriptores se regeneran desde `/kaggle/input` porque los CSV de las notebooks anteriores no forman parte de esta corrida.",
        "",
        "## 2. Figuras refinadas",
        "",
        STATE.get("duration_decision", ""),
        "",
    ]
    for record in figures.itertuples(index=False):
        lines.append(f"- {record.figure_id}. {record.title} Estado: {record.status}.")
    lines.extend(["", "## 3. Tablas refinadas", ""])
    for record in STATE["table_selection"].itertuples(index=False):
        lines.append(f"- {record.table_id}. {record.title} Estado: {record.status}.")
    lines.extend(["", "## 4. Cinco hallazgos centrales", ""])
    for record in findings.itertuples(index=False):
        lines.extend([
            f"### {record.finding_id}",
            "",
            f"EVIDENCIA: {record.observation} {record.quantitative_evidence}",
            "",
            f"INTERPRETACIÓN: {record.interpretation}",
            "",
            f"IMPLICACIÓN: {record.implication_for_preparation}",
            "",
        ])
    lines.extend([
        "## 5. Variables que justifican análisis posterior",
        "",
        "RMS, ZCR, frecuencia dominante, centroide, ancho de banda, roll-off, flujo, flatness y los 13 MFCC muestran variación medible en la muestra. Justificar un análisis posterior significa que conviene volver a examinarlas durante la preparación. No significa que hayan quedado elegidas para un modelo.",
        "",
        "## 6. Problemas metodológicos detectados",
        "",
        "- La duración de AudioSet-EV v2 no se lee bien si se la pone en el mismo eje y con la misma unidad que los 13 tracks de A3Siren-Recordings.",
        "- LSSiren contiene varias frecuencias de muestreo, así que un centroide en hertz no describe la misma banda en todos sus archivos.",
        "- Los ocho canales de A3Siren-Recordings pertenecen a una toma. Contarlos como 104 observaciones inflaría cualquier resumen.",
        "- Una diferencia de medianas entre corpus no demuestra domain shift.",
        "",
        "## 7. Implicaciones para Capítulo 4",
        "",
        "Cada fila de `Hallazgos_Puente_Capitulo_4.csv` es un hallazgo, su evidencia y una acción posible. Ninguna acción se ejecuta aquí.",
        "",
        "## 8. Figuras recomendadas para el cuerpo principal",
        "",
    ])
    for record in figures[figures["status"] == "PRINCIPAL"].itertuples(index=False):
        lines.append(f"- {record.figure_id}: {record.title}")
    lines.extend(["", "## 9. Figuras recomendadas para anexos", ""])
    for record in figures[figures["status"] != "PRINCIPAL"].itertuples(index=False):
        lines.append(f"- {record.figure_id} ({record.status}): {record.title}")
    lines.extend([
        "",
        "## 10. Estado del EDA",
        "",
        "La exploración queda resumida para la redacción del capítulo 3. No hay modelo, ni partición definitiva, ni característica declarada como definitiva. El paso siguiente es evaluar, en el capítulo de preparación, las acciones listadas en el puente.",
        "",
    ])
    report_path = OUTPUT_ROOT / "reports" / "refined_eda_summary.md"
    report_path.write_text("\n".join(lines), encoding="utf-8")
    print(f"escrito: {report_path}")
    sample = STATE["sample"]
    inventory = STATE["inventory"]
    missing = [path for path in sample["absolute_path"] if not Path(path).exists()]
    invalid = int((sample["quality_status"] != "OK").sum())
    secondary = sample[(sample["dataset_id"] != "a3siren") & (~sample["is_hash_primary"])]
    a3_without_track = sample[(sample["dataset_id"] == "a3siren") & (sample["track_id"].astype(str) == "")]
    input_writes = list(OUTPUT_ROOT.resolve().parents)
    checks = {
        "invalid_in_sample": invalid,
        "secondary_duplicates_in_sample": int(len(secondary)),
        "a3_without_track_id": int(len(a3_without_track)),
        "missing_files": len(missing),
        "output_is_input": any(path == INPUT_ROOT.resolve() for path in input_writes),
        "seed": SEED,
        "findings": int(len(STATE["findings"])),
        "report_exists": report_path.exists(),
        "inventory_files": int(len(inventory)),
    }
    print(json.dumps(checks, ensure_ascii=False))
    if invalid or len(secondary) or len(a3_without_track) or missing or checks["output_is_input"] or checks["findings"] != 5 or not checks["report_exists"]:
        raise AssertionError(f"Validación incompleta: {checks}")
    figures = STATE["figure_selection"]
    print("==============================================")
    print("NOTEBOOK 01.1 — REFINED EVIDENCE COMPLETED")
    print("==============================================")
    print()
    print("Refined figures:")
    print(len(list((OUTPUT_ROOT / "plots").glob("*.png"))))
    print()
    print("Refined tables:")
    print(len(list((OUTPUT_ROOT / "tables").glob("*.csv"))))
    print()
    print("Central findings:")
    print(5)
    print()
    print("Thesis principal figures:")
    print(int((figures["status"] == "PRINCIPAL").sum()))
    print()
    print("Thesis secondary figures:")
    print(int((figures["status"] == "SECUNDARIA").sum()))
    print()
    print("Appendix figures:")
    print(int((figures["status"] == "ANEXO").sum()))
    print()
    print("Bridge to Chapter 4:")
    print(len(STATE["bridge"]))
    print("==============================================")

section_report()
